# Quality Prediction in an Iron Ore Mining Process**Goal:** predict the **% Silica in the iron ore concentrate** (the impurity left after froth flotation) from industrial process measurements.This notebook walks through the full pipeline:`Data Loading → Inspection → Cleaning → Timestamp Processing → EDA → Feature Preparation → Chronological Split → Regression Models → Prediction → Evaluation → Feature Importance`It reuses the functions in `src/`, so the notebook and the scripts can never disagree.

In [ ]:
import sys, warningsfrom pathlib import Pathwarnings.filterwarnings("ignore")# Make src/ importable whether the notebook runs from notebooks/ or the project root.PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parentsys.path.append(str(PROJECT_ROOT / "src"))import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snssns.set_theme(style="whitegrid")%matplotlib inlineimport data_preprocessing as dpimport train_model as tmimport evaluate_model as emprint("Project root:", PROJECT_ROOT)

## 1. Data LoadingThe dataset path is defined once in `src/data_preprocessing.py`. If the file is missing, a clear error explains where to put it — we never fabricate data.

In [ ]:
raw = dp.load_raw_data(dp.DEFAULT_DATA_PATH)raw.head()

## 2. Data InspectionShape, columns, dtypes, missing values, duplicates, statistics, and identification of the timestamp and target columns.

In [ ]:
summary = dp.inspect_data(raw)summary

## 3. Data Cleaning + 4. Timestamp ProcessingEach step is simple and explainable:1. Parse the timestamp, sort chronologically.2. Drop exactly-duplicated rows.3. Force value columns to numeric (the raw Kaggle export uses comma decimals like `55,2`).4. Blank physically impossible values (percentages outside 0–100, negative flows/levels).5. **Align sampling frequencies** — process signals are logged every 20 s but the lab target only once per hour, so sub-hourly data is averaged to hourly means.6. Interpolate remaining gaps over time; never impute the target itself.7. Extract `hour`, `day`, `day_of_week`, `month`.

In [ ]:
clean = dp.clean_data(raw)clean.head()

## 5. Exploratory Data AnalysisFive plots, saved to `outputs/plots/`.

In [ ]:
corr_with_target = dp.run_eda(clean)corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index).head(10)

In [ ]:
from IPython.display import Image, displayfor name in ["silica_distribution.png", "silica_over_time.png",             "feature_target_relationship.png", "feature_target_correlation.png"]:    display(Image(filename=str(dp.PLOTS_DIR / name)))

### The key EDA finding`% Iron Concentrate` has a correlation near **−0.80** with `% Silica Concentrate`. That is expected chemistry: the concentrate is mostly iron and silica, so more of one means less of the other.But both numbers come from the **same hourly laboratory assay**. If we already had the iron result, we would already have the silica result — so using it as an input inflates the score without being useful in practice. That motivates running two experiments.

In [ ]:
clean.to_csv(dp.PROCESSED_PATH, index=False)print("Saved:", dp.PROCESSED_PATH)

## 6. Feature Preparation- `X` = process variables (feed grades, reagent flows, pulp properties, column air flows and levels) + `hour`, `day_of_week`- `y` = `% Silica Concentrate``day` and `month` are dropped from the model inputs: with a chronological split the test months never appear in training, so those columns would only invite spurious extrapolation.Two experiments:- **A `with_iron`** — includes `% Iron Concentrate` (demonstrates the leakage effect)- **B `without_iron`** — process variables only (the realistic, deployable model)

In [ ]:
X_a, y_a, feats_a = tm.build_feature_matrix(clean, include_iron=True)X_b, y_b, feats_b = tm.build_feature_matrix(clean, include_iron=False)print(f"Experiment A: {len(feats_a)} features")print(f"Experiment B: {len(feats_b)} features")feats_b

## 7. Train / Test Split — chronological, never shuffledThe earliest **80 %** of the timeline trains; the latest **20 %** tests.Shuffling randomly would place a reading from 14:00 in training and 14:20 in testing. Consecutive samples in a continuous process are nearly identical, so the model would be graded on rows it has effectively already memorised. A forward-in-time split matches how the model would really be used: fit on the past, predict the future.

In [ ]:
Xtr, Xte, ytr, yte, dtr, dte = tm.chronological_split(X_b, y_b, clean["date"])

## 8. Regression Models| Model | Why it is here ||---|---|| **Linear Regression** | The baseline. Assumes silica is a weighted sum of the inputs. Scaled so coefficients are comparable. || **Random Forest** | Averaged decision trees. Captures non-linearities and interactions; gives feature importances. || **Gradient Boosting** | Trees built sequentially, each fixing the previous errors. The "basic improvement" step. |

In [ ]:
res_a, bundle_a = tm.run_experiment(clean, include_iron=True,  label="with_iron")res_b, bundle_b = tm.run_experiment(clean, include_iron=False, label="without_iron")metrics = pd.concat([res_a, res_b], ignore_index=True)metrics.to_csv(dp.OUTPUT_DIR / "validation_metrics.csv", index=False)metrics

## 9. Evaluation- **MAE** — average absolute error, in percentage points of silica. Easy to explain to a plant engineer.- **RMSE** — squares the errors first, so big misses are punished harder.- **R²** — fraction of the variation in silica the model explains. 1.0 is perfect, 0.0 is no better than always guessing the mean, and negative is worse than the mean.`Train_R2` is shown alongside to expose overfitting.

In [ ]:
metrics.sort_values(["Experiment", "R2"], ascending=[True, False])

## 10. Feature Importance

In [ ]:
bundles = {"with_iron": bundle_a, "without_iron": bundle_b}importances = tm.feature_importance_report(bundles)display(Image(filename=str(dp.PLOTS_DIR / "feature_importance.png")))

## 11. Prediction AnalysisPredictions from the selected model on the held-out test window, plus residual diagnostics.

In [ ]:
import jsonbest_b = res_b.sort_values("R2", ascending=False).iloc[0](dp.OUTPUT_DIR / "selected_model.json").write_text(json.dumps({    "selected_experiment": "without_iron",    "selected_model": best_b["Model"],    "reason": "Process-only model: % Iron Concentrate comes from the same hourly lab assay as the target.",    "test_MAE": float(best_b["MAE"]), "test_RMSE": float(best_b["RMSE"]), "test_R2": float(best_b["R2"]),}, indent=2))preds = em.main()preds.head(10)

In [ ]:
display(Image(filename=str(dp.PLOTS_DIR / "actual_vs_predicted.png")))display(Image(filename=str(dp.PLOTS_DIR / "residual_analysis.png")))

## 12. OPTIONAL — Forecasting experiment*Clearly labelled as optional / exploratory, not part of the core deliverable.*The brief asks how many hours ahead silica can be predicted. Using only information available at time *t* (process variables + lags of past silica), we try to predict the silica measured at *t + h* for h = 1, 2 and 4 hours, and compare against a **persistence baseline** ("assume silica stays where it is"). A forecast is only worth anything if it beats doing nothing.

In [ ]:
forecast = tm.forecasting_experiment(clean)forecast

## 13. ConclusionsRun the cells above for the exact figures; the headline findings are:1. **With `% Iron Concentrate`, prediction looks easy** — but that variable is the target's lab-assay twin, so the score is optimistic and not usable for real-time control.2. **Without it, the process variables alone explain only a small share of the hourly variation.** This is an honest negative result, not a bug: reagent flows, air flows and column levels are held near setpoints, so they simply do not vary enough to explain hour-to-hour swings in silica.3. **Non-linear models beat the linear baseline** in the realistic experiment, confirming flotation behaviour is not a weighted sum of inputs.4. **The persistence baseline is strong at short horizons** — the most useful single predictor of the next silica reading is the current one.See `README.md` for the full write-up, limitations and future improvements.